# Download RedPajama Books + ArXiv
Streams `togethercomputer/RedPajama-Data-1T` book and arxiv splits.
Saves into separate `data_books/` and `data_arxiv/` folders, same `.txt` + `<|endoftext|>` format.
Resumable. Books ~100 GB, ArXiv ~115 GB.
Run `consolidate_data` -> `tokenizer_pipeline` -> `pre_train` after.

In [ ]:
!pip install -q "datasets<3.0.0"

In [ ]:
import os
from google.colab import drive
from datasets import load_dataset

drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/synapse/datasets_pretrain"
SEPARATOR = "\n<|endoftext|>\n"

# (config_name, output_subdir, file_prefix, docs_per_file)
SOURCES = [
    ("book",  "data_books", "books_part", 2_000),
    ("arxiv", "data_arxiv", "arxiv_part", 5_000),
]

# Books are long (whole books) -> fewer per file. ArXiv papers are medium-long.
MAX_DOCS_PER_SOURCE = 100_000_000  # sentinel; exits when stream ends
MIN_LENGTH = 200                    # skip near-empty docs

In [ ]:
for config_name, subdir, prefix, docs_per_file in SOURCES:
    out_dir = os.path.join(BASE_DIR, subdir)
    os.makedirs(out_dir, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"Source: RedPajama '{config_name}' -> {out_dir}")
    print(f"{'='*60}")

    # Resume detection
    existing = sorted([
        f for f in os.listdir(out_dir)
        if f.startswith(f"{prefix}_") and f.endswith(".txt")
    ])
    if existing:
        last_num = int(existing[-1].replace(f"{prefix}_", "").replace(".txt", ""))
        docs_done = (last_num + 1) * docs_per_file
        print(f"  Found {len(existing)} existing files (~{docs_done:,} docs)")
        print(f"  Resuming from doc {docs_done:,}...")
    else:
        docs_done = 0

    try:
        ds = load_dataset(
            "togethercomputer/RedPajama-Data-1T",
            config_name,
            split="train",
            streaming=True,
            trust_remote_code=True,
        )
    except Exception as e:
        print(f"  Failed to load '{config_name}': {e}")
        continue

    buf = []
    file_idx = len(existing)
    src_chars = 0
    src_docs = 0
    skipped = 0

    for i, example in enumerate(ds):
        if i >= MAX_DOCS_PER_SOURCE:
            break
        if i < docs_done:
            if i % 100_000 == 0 and i > 0:
                print(f"  Skipping... {i:,}/{docs_done:,}")
            continue

        text = example.get("text", "").strip()
        if len(text) < MIN_LENGTH:
            skipped += 1
            continue

        buf.append(text)
        src_docs += 1

        if len(buf) >= docs_per_file:
            path = os.path.join(out_dir, f"{prefix}_{file_idx:04d}.txt")
            content = SEPARATOR.join(buf)
            with open(path, "w", encoding="utf-8") as f:
                f.write(content)
            size_mb = os.path.getsize(path) / 1024 / 1024
            src_chars += len(content)
            src_gb = src_chars / 1024 / 1024 / 1024
            print(f"  {path.split('/')[-1]}: {size_mb:.0f} MB | {src_docs:,} docs | {src_gb:.1f} GB | {skipped:,} skipped")
            buf = []
            file_idx += 1

    # Write remainder
    if buf:
        path = os.path.join(out_dir, f"{prefix}_{file_idx:04d}.txt")
        content = SEPARATOR.join(buf)
        with open(path, "w", encoding="utf-8") as f:
            f.write(content)
        src_chars += len(content)
        file_idx += 1

    src_gb = src_chars / 1024 / 1024 / 1024
    print(f"\n  {config_name}: {src_docs:,} docs, {file_idx} files, ~{src_gb:.1f} GB, {skipped:,} skipped")

print("\nAll done!")
print("Next: run consolidate_data -> tokenizer_pipeline -> pre_train")